# Kalomo Town Council — Financial & Revenue Data
### CSC4792 Mini Project — Financial/Revenue component (Faiz)

This notebook documents how the financial and revenue dataset for
**Kalomo Town Council** was scraped, extracted, and cleaned from the
council's official website (kalomocouncil.gov.zm), as required by the
CSC4792 mini-project brief (Issue #2: "Faiz: Scrape financial and
revenue data").

**What this dataset covers** (per the assignment requirements and
Issue #2's data points):
- Approved budgets (by fiscal year)
- Own Source Revenue (OSR) targets and actual collections
- Central Government transfers / cooperating partner funding
- Local Government Equalisation Fund (LGEF) allocations
- Actual expenditure vs. approved budget
- Local revenue sources: market fees, local taxes, fees and charges
- Budget performance percentages (`percent_of_target`)

**A note on method:** the council's `robots.txt` disallows automated
crawling. The scraping scripts here use plain `requests`, which does not
enforce robots.txt automatically, but this is disclosed here and in the
Data in Brief paper for transparency, and requests are rate-limited
(`time.sleep`) to avoid hammering the server.


## Step 1: Identify source pages

Financial information on the council's site is published as ordinary
news posts (WordPress `?p=NNNN` URLs) and as downloadable PDFs (budget
documents, investment profiles, CDF newsletters). These were located by
manually browsing the site and are listed in
`data/raw/financial_data/urls.txt`.

In [1]:
with open("../data/raw/financial_data/urls.txt") as f:
    print(f.read())

# Source pages containing Kalomo Town Council financial / revenue information
# One URL per line. Add more as you find them by browsing the site yourself.

https://www.kalomocouncil.gov.zm/?p=4687
https://www.kalomocouncil.gov.zm/?p=3782
https://www.kalomocouncil.gov.zm/?p=4672
https://www.kalomocouncil.gov.zm/wp-content/uploads/2025/11/Kalomo-town-Council-2.0-Investment-Profile-1-1.pdf
https://www.kalomocouncil.gov.zm/wp-content/uploads/2024/04/2024-1ST-QUARTER-CDF-NEWSLETTER-.pdf
https://www.mlgrd.gov.zm/wp-content/uploads/2023/07/KALOMO-TOWN-COUNCIL-1.pdf



## Step 2: Scrape the HTML pages and PDFs

Three scripts do the scraping:
- `scripts/scraping/scrape_financial_html.py` — downloads each HTML news
  page and extracts the title and main article text using BeautifulSoup.
- `scripts/scraping/scrape_financial_pdfs.py` — downloads each linked
  PDF and extracts its text using `pdfplumber`.
- `scripts/scraping/crawl_and_scrape_financial.py` — a broader crawler
  that follows internal links from the seed URLs in `urls.txt` to find
  additional budget/financial pages and PDFs that aren't linked directly
  from the pages we started with (e.g. `2025-Revised-Budget.pdf`,
  `2026-BUDGET-CONSULTATIVE-MEETING.pdf`). Its progress is checkpointed
  in `data/raw/financial_data/_crawl_state.json` so it can resume without
  re-downloading pages, and every URL it visits is logged in
  `all_urls_visited.txt` for full provenance.

All scraping saves its raw output into `data/raw/financial_data/` as CSV
files (`raw_scraped_articles.csv` and `raw_scraped_pdfs.csv`) and the
raw PDFs themselves are kept in `data/raw/financial_data/pdfs/`. This raw
stage deliberately keeps the full article/PDF text rather than numbers,
so nothing is lost before the extraction step.


In [2]:
import pandas as pd

raw_articles = pd.read_csv("../data/raw/financial_data/raw_scraped_articles.csv")
raw_articles[["source_url", "title"]]

,source_url,title
0,https://www.kalomocouncil.gov.zm/,Kalomo Town Council – Kalomo
1,https://www.kalomocouncil.gov.zm/?page_id=169,Mandate – Kalomo Town Council
2,https://www.kalomocouncil.gov.zm/?page_id=3964,INSTITUTIONAL MANAGEMENT – Kalomo Town Council
3,https://www.kalomocouncil.gov.zm/?page_id=4267,Department of Engineering Services – Kalomo To...
4,https://www.kalomocouncil.gov.zm/?page_id=4280,Department of Finance – Kalomo Town Council
...,...,...
87,https://www.kalomocouncil.gov.zm/?attachment_i...,WhatsApp Image 2024-01-18 at 23.07.24
88,https://www.kalomocouncil.gov.zm/?author=1,Author:justine.bwalya
89,https://www.kalomocouncil.gov.zm/?cat=5,Category:news updates
90,https://www.kalomocouncil.gov.zm/?p=1048,Expired Goods Seized


## Step 3: Extract structured financial figures

Two different extraction strategies are needed because council figures
appear in two very different formats in the raw text:

**1. Narrative figures** — plain sentences in news posts/meeting minutes, e.g.:

> "Kalomo Town Council adopted a total approved budget of K131,360,483
> for the 2026 financial year..."

`extract_narrative_records()` in `scripts/cleaning/clean_financial_data.py`
uses targeted regular expressions to pull out each K-amount and classify
it (e.g. `total_approved_budget`, `expenditure_actual`) based on the
surrounding wording.

**2. Table figures** — budget PDFs where `pdfplumber` flattens a table
into a single line of text with no column boundaries, e.g.:

```
Local Government Equalisation Fund 11,706,441 ...
Market fees 250,000 ...
Local Taxes 1,407,959 703,612 47
```

`extract_table_records()` handles these with its own set of patterns
(LGEF, market fees, local taxes, fees and charges), pulling the fiscal
year from the source filename/URL when it isn't stated in the row
itself. These table-based rules are noted as more fragile in the data
dictionary, since they depend on the exact column order the council
used in that particular document.

Both strategies are explicit, readable rules rather than generic NLP,
so every extracted number can be traced back to exactly why it was
classified that way — important for a small, council-specific dataset
where accuracy matters more than scale.

**A bug worth documenting:** an early version of the expenditure rule
silently failed to match because `pdfplumber` had inserted a line break
in the middle of the sentence ("total expenditure for\nthe period..."),
and a literal space in a regex does not match a newline. The fix,
applied in `extract_records_from_text()`, is to collapse all whitespace
(spaces, tabs, newlines) in the scraped text down to single spaces
*before* running any extraction rule. This is a good general lesson for
PDF-derived text: never assume a sentence stays on one line.


## Step 4: Clean and structure the data

Cleaning steps applied:
- Collapse whitespace in raw scraped text before extraction (see the bug
  note in Step 3) so line-wrapped sentences don't break the regex rules.
- Strip the `K` currency symbol and thousands separators, convert to
  numeric (float) values in ZMW.
- Classify each figure as a `target` (budgeted/projected) or `actual`
  (collected/spent) value.
- Compute `percent_of_target` automatically where both a target and a
  matching actual figure exist for the same fiscal year — this covers
  OSR, Central Government transfers, local taxes, fees and charges, and
  expenditure vs. approved budget.
- Remove exact duplicate records (same figure appearing on more than one
  page).
- Add a `notes` field for figures with important context (e.g. the OSR
  drop attributed to lower plot premium projections, or a flag that a
  figure came from a fragile flattened PDF table).


In [3]:
import subprocess
subprocess.run(["python3", "../scripts/cleaning/clean_financial_data.py"])

Saved 18 cleaned records to /home/faiz/Desktop/csc4792-kalomo-town-council/data/processed/db-unza26-csc4792-kalomo_town_council_financial_data.csv
           council_name  fiscal_year                    record_type   amount_zmw target_or_actual                                                                                               source_url scrape_date percent_of_target                                                                                                                                                               notes
0   Kalomo Town Council         2024             expenditure_actual   77076699.0           actual  http://www.kalomocouncil.gov.zm/wp-content/uploads/2025/12/community-Engagement-meeting-2025-Budget.pdf  2026-09-12              80.8                                                                                                                                                                    
1   Kalomo Town Council         2024          total_approve

CompletedProcess(args=['python3', '../scripts/cleaning/clean_financial_data.py'], returncode=0)

## Step 5: Final dataset preview

In [4]:
df = pd.read_csv(
    "../data/processed/db-unza26-csc4792-kalomo_town_council_financial_data.csv",
    sep="|"
)
df

,council_name,fiscal_year,record_type,amount_zmw,target_or_actual,source_url,scrape_date,percent_of_target,notes
0,Kalomo Town Council,2024,expenditure_actual,77076699.0,actual,http://www.kalomocouncil.gov.zm/wp-content/upl...,2026-09-12,80.8,NaN
1,Kalomo Town Council,2024,total_approved_budget,95399253.0,target,http://www.kalomocouncil.gov.zm/wp-content/upl...,2026-09-12,NaN,NaN
2,Kalomo Town Council,2025,central_govt_transfers_budget,135701875.0,target,https://www.kalomocouncil.gov.zm/?p=4687,2026-09-12,NaN,NaN
3,Kalomo Town Council,2025,fees_and_charges_actual,1260217.0,actual,http://www.kalomocouncil.gov.zm/wp-content/upl...,2026-09-12,68.0,Extracted from a flattened PDF budget table; v...
4,Kalomo Town Council,2025,fees_and_charges_actual,210000.0,actual,http://www.kalomocouncil.gov.zm/wp-content/upl...,2026-09-12,11.3,Extracted from a flattened PDF budget table; v...
5,Kalomo Town Council,2025,fees_and_charges_actual,200000.0,actual,http://www.kalomocouncil.gov.zm/wp-content/upl...,2026-09-12,10.8,Extracted from a flattened PDF budget table; v...
6,Kalomo Town Council,2025,fees_and_charges_budget,1854513.0,target,http://www.kalomocouncil.gov.zm/wp-content/upl...,2026-09-12,NaN,Extracted from a flattened PDF budget table; v...
7,Kalomo Town Council,2025,fees_and_charges_budget,200000.0,target,http://www.kalomocouncil.gov.zm/wp-content/upl...,2026-09-12,NaN,Extracted from a flattened PDF budget table; v...
8,Kalomo Town Council,2025,lgef_disbursement,11706441.0,target,http://www.kalomocouncil.gov.zm/wp-content/upl...,2026-09-12,NaN,Extracted from a flattened PDF budget table; v...
9,Kalomo Town Council,2025,local_taxes_actual,713855.0,actual,http://www.kalomocouncil.gov.zm/wp-content/upl...,2026-09-12,47.8,Extracted from a flattened PDF budget table; v...


## Step 6: Output format compliance check

Per the assignment brief and Issue #2's acceptance criteria, the final
CSV must:
- Be in CSV format ✅
- Use `|` as the column separator ✅ (`sep="|"` used above and in export)
- Follow the naming convention `db-unza26-csc4792-[description].csv` ✅
  → `db-unza26-csc4792-kalomo_town_council_financial_data.csv`
- Have numeric amounts with no stray currency symbols/commas ✅
  (`to_number()` strips `K`, commas, and spaces before converting to float)
- Record a `source_url` and `scrape_date` for every row ✅
- Be reflected in `docs/DATA_DICTIONARY.md` ✅


## Next steps / limitations

This notebook now covers approved budgets, LGEF, expenditure, and local
revenue (market fees, local taxes, fees and charges) across fiscal years
2024–2026, sourced from both prose (news posts, meeting minutes) and
flattened PDF budget tables.

Known limitations to flag in the Data in Brief paper:
- Table-based rows (`_budget`/`_actual` figures pulled from flattened
  PDF tables) are more fragile than prose-based rows — always spot-check
  a few against the source PDF listed in `source_url`.
- Some fiscal years only have a `target` or only an `actual` figure for
  a given record type (no matching pair yet), so `percent_of_target` is
  blank for those rows — this is expected, not a bug.
- To make the dataset richer, extend `urls.txt` / let the crawler run
  further to pick up additional quarterly budget performance reports,
  older years' budgets, and CDF-specific disbursement reports, then add
  matching extraction rules to `clean_financial_data.py` for any new
  phrasing patterns found.
